In [15]:
!pip install pyspark -q

In [16]:
import sqlite3, json, requests, pandas as pd

df = pd.read_csv("week12_transactions.csv")
conn = sqlite3.connect("momo.db")
df.to_sql("transactions", conn, if_exists="replace", index=False)
print(df.shape)

(2000, 7)


In [17]:
# (a) 5 largest send transactions above 100,000 RWF
q = """SELECT txn_id, district, amount_rwf
       FROM transactions
       WHERE txn_type = 'send' AND amount_rwf > 100000
       ORDER BY amount_rwf DESC
       LIMIT 5"""
print(pd.read_sql(q, conn))

# (b) flagged count (checkpoint: 11)
print(pd.read_sql("SELECT COUNT(*) AS flagged_count FROM transactions WHERE flagged = 1", conn))

   txn_id    district  amount_rwf
0  T01708    Kicukiro      140000
1  T00364        Huye      139000
2  T00895     Muhanga      139000
3  T00899  Nyarugenge      139000
4  T01314      Gasabo      139000
   flagged_count
0             11


In [18]:
# (a) per txn_type
q = """SELECT txn_type, COUNT(*) AS n, SUM(amount_rwf) AS total, AVG(amount_rwf) AS avg_amount
       FROM transactions GROUP BY txn_type ORDER BY total DESC"""
print(pd.read_sql(q, conn))
# Answer: 'send' moves the most money (~62.0M RWF)

# (b) per district
q = """SELECT district, SUM(amount_rwf) AS total
       FROM transactions GROUP BY district ORDER BY total DESC"""
print(pd.read_sql(q, conn))
# Answer: Muhanga leads (~21.1M RWF)

  txn_type    n     total    avg_amount
0     send  868  62049000  71485.023041
1  paybill  330  29970000  90818.181818
2  receive  489  27024000  55263.803681
3  airtime  313    833000   2661.341853
     district     total
0     Muhanga  21076000
1      Gasabo  20738000
2        Huye  18736000
3  Nyarugenge  18091000
4    Kicukiro  11338000
5     Musanze  10490500
6      Rubavu  10110500
7   Nyagatare   9296000


In [19]:
# (a) SQL
q = "SELECT hour, COUNT(*) AS n FROM transactions GROUP BY hour ORDER BY n DESC"
sql_hours = pd.read_sql(q, conn)
print(sql_hours.head())

# (b) Pandas
pd_hours = df["hour"].value_counts()
print(pd_hours.head())

print("Agree:", sql_hours.loc[0, "hour"] == pd_hours.index[0] and sql_hours.loc[0, "n"] == pd_hours.iloc[0])
# Busiest hour: 20 with 185 transactions

   hour    n
0    20  185
1    17  183
2    18  176
3    12  170
4    11  169
hour
20    185
17    183
18    176
12    170
11    169
Name: count, dtype: int64
Agree: True


In [20]:
import json

# Sample data. Replace this whole string with the one from your lab notebook.
json_str = '''{"city": "Kigali", "readings": [21.5, 23.0, 25.5, 27.0, 24.5]}'''

d = json.loads(json_str)
print(d)

# Find the city (first text value) and the readings (first list of numbers)
city = next(v for v in d.values() if isinstance(v, str))
readings = next(v for v in d.values() if isinstance(v, list))

print("City:", city)                                              # (a)
print("Number of readings:", len(readings))                       # (b)
print("Average:", round(sum(readings) / len(readings), 2))        # (c)

{'city': 'Kigali', 'readings': [21.5, 23.0, 25.5, 27.0, 24.5]}
City: Kigali
Number of readings: 5
Average: 24.3


In [21]:
try:
    url = ("https://api.open-meteo.com/v1/forecast"
           "?latitude=-1.9441&longitude=30.0619"
           "&hourly=temperature_2m&forecast_days=1&timezone=Africa%2FKigali")
    data = requests.get(url, timeout=10).json()
except Exception:
    with open("week12_api_sample.json") as f:
        data = json.load(f)

times = data["hourly"]["time"]
temps = data["hourly"]["temperature_2m"]

print("Hours:", len(temps))
print("Max:", max(temps), " Min:", min(temps))
print("Hottest hour:", times[temps.index(max(temps))])

Hours: 24
Max: 25.8  Min: 14.6
Hottest hour: 2026-11-25T12:00


In [22]:
weather = pd.DataFrame({"time": times, "temp_c": temps})
print(weather["temp_c"].describe())
print("Hours >= 25 C:", (weather["temp_c"] >= 25).sum())   # sample file: 7

count    24.000000
mean     19.808333
std       4.028584
min      14.600000
25%      16.375000
50%      19.150000
75%      23.075000
max      25.800000
Name: temp_c, dtype: float64
Hours >= 25 C: 5


In [23]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("week12").master("local[*]").getOrCreate()

In [24]:
sdf = spark.read.csv("week12_transactions.csv", header=True, inferSchema=True)
print("Rows:", sdf.count())   # 2000
sdf.printSchema()
sdf.show(5)

Rows: 2000
root
 |-- txn_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)

+------+----------+----+----------+--------+----------+-------+
|txn_id|      date|hour|  district|txn_type|amount_rwf|flagged|
+------+----------+----+----------+--------+----------+-------+
|T00001|2026-11-01|   8|    Gasabo|    send|      1000|      0|
|T00002|2026-11-09|  12|Nyarugenge|    send|     25000|      0|
|T00003|2026-11-19|   8|   Muhanga|    send|     51000|      0|
|T00004|2026-11-01|  17|      Huye|    send|     79000|      0|
|T00005|2026-11-15|  12|      Huye|    send|    109000|      0|
+------+----------+----+----------+--------+----------+-------+
only showing top 5 rows


In [25]:
print(sdf.filter(sdf.amount_rwf > 100000).count())   # checkpoint: 451
sdf.filter(sdf.flagged == 1).show(5)

451
+------+----------+----+--------+--------+----------+-------+
|txn_id|      date|hour|district|txn_type|amount_rwf|flagged|
+------+----------+----+--------+--------+----------+-------+
|T00279|2026-11-29|  19| Muhanga|    send|    127000|      1|
|T00326|2026-11-07|  22|    Huye|    send|    127000|      1|
|T00536|2026-11-27|  17|  Gasabo|    send|    127000|      1|
|T00655|2026-11-30|  17|  Rubavu|    send|    134000|      1|
|T00697|2026-11-01|   9| Muhanga|    send|    127000|      1|
+------+----------+----+--------+--------+----------+-------+
only showing top 5 rows


In [26]:
# (a)
sdf.groupBy("txn_type").sum("amount_rwf").show()

# (b)
dist = sdf.groupBy("district").sum("amount_rwf").toPandas()
print(dist.sort_values("sum(amount_rwf)", ascending=False))

# (c) Yes, the Spark totals match the SQL totals (send ~62.0M, Muhanga ~21.1M).
#     Same data and same GROUP BY logic, just a different engine.

+--------+---------------+
|txn_type|sum(amount_rwf)|
+--------+---------------+
| paybill|       29970000|
| airtime|         833000|
| receive|       27024000|
|    send|       62049000|
+--------+---------------+

     district  sum(amount_rwf)
4     Muhanga         21076000
5      Gasabo         20738000
6        Huye         18736000
1  Nyarugenge         18091000
0    Kicukiro         11338000
2     Musanze         10490500
7      Rubavu         10110500
3   Nyagatare          9296000


In [27]:
sdf.createOrReplaceTempView("transactions")
spark.sql("""SELECT txn_type, COUNT(*) AS n, SUM(amount_rwf) AS total, AVG(amount_rwf) AS avg_amount
             FROM transactions GROUP BY txn_type ORDER BY total DESC""").show()

# Why powerful: SQL is portable and declarative. You say WHAT you want and the engine decides HOW,
# so the same query works on a laptop (SQLite) or a huge cluster (Spark) without rewriting the logic.

+--------+---+--------+------------------+
|txn_type|  n|   total|        avg_amount|
+--------+---+--------+------------------+
|    send|868|62049000| 71485.02304147465|
| paybill|330|29970000| 90818.18181818182|
| receive|489|27024000|  55263.8036809816|
| airtime|313|  833000|2661.3418530351437|
+--------+---+--------+------------------+



1. **SQL**: the data already lives in PostgreSQL, so a GROUP BY runs next to the data without copying it out.
2. **Pandas**: 40,000 rows fit in memory, and Pandas with matplotlib/scikit-learn covers cleaning, charts and regression.
3. **Spark**: 40 TB on a cluster is far too big for one machine; Spark distributes the work across the nodes where the data sits.
4. **API**: only an API delivers fresh data on demand with no one uploading a file.